In [8]:
import pandas as pd
import requests
from io import StringIO
import plotly.express as px

url = "https://en.wikipedia.org/wiki/List_of_countries_with_McDonald%27s_restaurants"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# baixar página
resposta = requests.get(url, headers=headers)

# pegar tabelas
tabelas = pd.read_html(StringIO(resposta.text))

df = tabelas[0].copy()

# padronizar nomes de colunas
df.columns = df.columns.str.strip()

# encontrar colunas automaticamente
country_col = [c for c in df.columns if "Country" in c][0]
mcd_col = [c for c in df.columns if "operating" in c][0]

# limpar nomes de países
df[country_col] = df[country_col].str.replace(r"\s*\(.*?\)", "", regex=True)

# limpar números
df[mcd_col] = df[mcd_col].astype(str)
df[mcd_col] = df[mcd_col].str.replace(r"\[.*?\]", "", regex=True)
df[mcd_col] = df[mcd_col].str.replace(",", "")
df[mcd_col] = pd.to_numeric(df[mcd_col], errors="coerce")

# remover NaN
df = df.dropna(subset=[mcd_col])

# ordenar
df = df.sort_values(by=mcd_col, ascending=False)

# mapa
fig = px.choropleth(
    df,
    locations=country_col,
    locationmode="country names",
    color=mcd_col,
    color_continuous_scale="YlOrRd",
    title="Distribuição de McDonald's pelo mundo"
)

import plotly.io as pio
pio.renderers.default = "browser"
fig.show()

C:\Users\marii\AppData\Local\Temp\ipykernel_13428\718120239.py:43: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(
